# Logistic Regression Implementation

In [ ]:
import numpy as mp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.datasets import make_classification

In [ ]:
# create
X,y=make_classification(
    n_samples=10000,
    n_features=10,
    n_classes=2,
    random_state=42,
)

In [ ]:
X=pd.DataFrame(X)
X

In [ ]:
y

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,random_state=42,test_size=0.25)

In [ ]:
X_test

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
logisticRegression = LogisticRegression()

In [ ]:
logisticRegression.fit(X_train, y_train)

In [ ]:
y_pred = logisticRegression.predict(X_test)
y_pred

In [ ]:
logisticRegression.predict_proba(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
score = accuracy_score(y_test,y_pred)
print(score)
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(classification_report(y_test, y_pred ))

## Hyperparameter Tuning

---

## 1. What is a hyperparameter?

In machine learning, **hyperparameters** are settings that **control the behavior of your model**, but they are **not learned from the data**.

Examples:

| Model          | Hyperparameter  | Description                    |
| -------------- | --------------- | ------------------------------ |
| Decision Tree  | `max_depth`     | Maximum depth of the tree      |
| Random Forest  | `n_estimators`  | Number of trees in the forest  |
| Neural Network | `learning_rate` | Step size for updating weights |
| SVM            | `C`, `kernel`   | Regularization and kernel type |

Contrast this with **parameters**, like weights in a neural network, which are **learned from the training data**.

---

## 2. Why tune hyperparameters?

Different hyperparameter settings can drastically change your model’s performance.

* Too low or high learning rate → slow convergence or divergence
* Too deep tree → overfitting, too shallow → underfitting

Hyperparameter tuning is about **finding the “best” combination** for your dataset.

---

## 3. Common methods for hyperparameter tuning

### A. Grid Search

* **Idea**: Try every combination of a predefined set of hyperparameter values.
* **Pros**: Exhaustive, simple.
* **Cons**: Computationally expensive for large grids.

**Example in Python (sklearn):**

```python
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20]
}

clf = RandomForestClassifier()
grid_search = GridSearchCV(clf, param_grid, cv=5)
grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
```

---

### B. Random Search

* **Idea**: Randomly sample hyperparameter combinations.
* **Pros**: Often finds good results faster than grid search.
* **Cons**: Might miss optimal settings.

**Example:**

```python
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    'n_estimators': randint(50, 200),
    'max_depth': randint(5, 20)
}

rand_search = RandomizedSearchCV(clf, param_dist, n_iter=10, cv=5)
rand_search.fit(X_train, y_train)

print(rand_search.best_params_)
```

---

### C. Bayesian Optimization

* **Idea**: Uses past evaluations to choose the next promising hyperparameters.
* **Pros**: More efficient than random/grid search for expensive models.
* **Cons**: More complex to implement.
* Tools: `Optuna`, `Hyperopt`, `scikit-optimize`.

---

### D. Manual Search

* **Idea**: Adjust hyperparameters based on intuition or trial and error.
* **Pros**: Sometimes fastest if you know your model well.
* **Cons**: Not systematic.

---

## 4. How the tuning process works

1. **Choose hyperparameters to tune** (like `learning_rate`, `n_estimators`).
2. **Define a range of values** (grid) or a distribution (random).
3. **Train the model** on training data for each combination.
4. **Evaluate performance** using cross-validation or a validation set.
5. **Select the best hyperparameters** based on metrics (accuracy, RMSE, F1-score, etc.).
6. **Retrain the model** on full training data using the best hyperparameters.

---

### ⚡ Tips

* Use **cross-validation** to avoid overfitting during tuning.
* Don’t tune too many hyperparameters at once — focus on the most important ones.
* Start coarse, then fine-tune around the best values.

---



In [ ]:
model = LogisticRegression()
# now we define the parameters we want to mess around with
# refer to the model's documentation to see the options and
# select the ones we want to search around in, for eg:
penalty = ['l1','l2','elasticnet']
c_values = [100,10,1.0,0.1,0.01]
solver = ['newton-cg', 'lbfgs', 'liblinear','sag','saga']
max_iter = [200,400,600,800,1000]

In [ ]:
params = dict(penalty = penalty, C = c_values, solver=solver, max_iter = max_iter)
# params = dict(penalty = penalty, C = c_values, solver=solver)

In [ ]:
param_grid = [
    # L1 penalty: only liblinear and saga
    {
        'penalty': ['l1'],
        'solver': ['liblinear', 'saga'],
        'C': [0.01, 0.1, 1, 10, 100],
        'max_iter': max_iter
    },
    # L2 penalty: works with most solvers
    {
        'penalty': ['l2'],
        'solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'],
        'C': [0.01, 0.1, 1, 10, 100],
        'max_iter': max_iter
    },
    # ElasticNet: only saga, needs l1_ratio
    {
        'penalty': ['elasticnet'],
        'solver': ['saga'],
        'l1_ratio': [0.3, 0.5, 0.7],  # mix of L1/L2
        'C': [0.01, 0.1, 1, 10, 100],
        'max_iter': max_iter
    }
]


In [ ]:
from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold()

In [ ]:
#GridSearchCV
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(estimator= model,
                    param_grid= param_grid,
                    scoring='f1',
                    cv=cv,
                    n_jobs=-1
                    )

In [ ]:
print(grid)

In [ ]:
grid.fit(X_train,y_train)

In [ ]:
grid.best_params_

In [ ]:
grid.best_score_

In [ ]:
y_pred = grid.predict(X_test)

In [ ]:
score = accuracy_score(y_test,y_pred)
print(score)
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(classification_report(y_test, y_pred ))

In [ ]:
# for more optimal results
scoring = ['accuracy', 'f1', 'roc_auc']

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring=scoring,
    refit='f1',  # <-- important!
    cv=cv,
    n_jobs=-1
)


In [ ]:
grid.fit(X_train,y_train)

In [ ]:
y_pred = grid.predict(X_test)

In [ ]:
score = accuracy_score(y_test,y_pred)
print(score)
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(classification_report(y_test, y_pred ))

In [ ]:
results = pd.DataFrame(grid.cv_results_)

# show hyperparameters + all metrics
results[['params', 'mean_test_accuracy', 'mean_test_f1', 'mean_test_roc_auc']]


In [ ]:
grid.cv_results_['mean_test_accuracy']  # Accuracy
grid.cv_results_['mean_test_f1']        # F1-score
grid.best_estimator_                    # model optimized for F1


In [ ]:
from sklearn.metrics import make_scorer, f1_score, accuracy_score, roc_auc_score

def custom_scoring(y_true, y_pred):
    f1 = f1_score(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_pred)
    # Combine scores, e.g., using a weighted sum
    return 0.5 * f1 + 0.3 * accuracy + 0.2 * roc_auc

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring=make_scorer(custom_scoring),
    cv=cv,
    n_jobs=-1
)


In [ ]:
print(grid)

In [ ]:
grid.fit(X_train,y_train)

In [ ]:
y_pred = grid.predict(X_test)
score = accuracy_score(y_test,y_pred)
print(score)
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(classification_report(y_test, y_pred ))

In [ ]:
grid.best_score_
y_pred = grid.predict(X_test)
score = accuracy_score(y_test,y_pred)
print(score)
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(classification_report(y_test, y_pred ))

### Random Search CV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
param_grid

In [ ]:
model = LogisticRegression()
randomCV = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    cv=5,
    n_jobs=-1,
    scoring='accuracy'
)

In [ ]:
randomCV.fit(X_train,y_train)

In [ ]:
grid.best_score_
y_pred = grid.predict(X_test)
score = accuracy_score(y_test,y_pred)
print(score)
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(classification_report(y_test, y_pred ))